In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/train_data.csv')

/tmp/ipykernel_4938/3655778083.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/drive/MyDrive/train_data.csv')


### 1. Exploratory Data Analysis (EDA) for Feature Selection
We will analyze the following features: `SOTagsHaveWorkedWith`, `SOTagsWantToWorkWith`, `SOTagsAdmired`, `SOTagsHaveEntry`, `SOTagsWant Entry`, `SOAccount`, `SOVisitFreq`, `SOPartFreq`, `SOComm`, `SOFriction`, and the `SO_Actions` group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Define the list of features to evaluate
my_features = [
    'SOTagsHaveWorkedWith', 'SOTagsWantToWorkWith', 'SOTagsAdmired',
    'SOTagsHaveEntry', 'SOTagsWant Entry', 'SOAccount', 'SOVisitFreq',
    'SOPartFreq', 'SOComm', 'SOFriction', 'SO_Actions_1', 'SO_Actions_16',
    'SO_Actions_3', 'SO_Actions_4', 'SO_Actions_6', 'SO_Actions_7',
    'SO_Actions_15', 'SO_Actions_15_TEXT'
]

# Find the target variable - assuming 'JobSat' based on standard SO survey schema
target = 'JobSat' if 'JobSat' in df.columns else None
if not target:
    # Fallback to look for column with 'Satisfaction' in name
    target = [c for c in df.columns if 'Sat' in c][0]

print(f"Target column identified as: {target}")

# 1. Check for Missing Values
missing_info = df[my_features].isnull().mean() * 100
print("\nPercentage of missing values per feature:")
print(missing_info)

# 2. Check for Unique Values (Cardinality)
unique_info = df[my_features].nunique()
print("\nUnique values per feature:")
print(unique_info)

Target column identified as: JobSatLevel

Percentage of missing values per feature:
SOTagsHaveWorkedWith    63.480858
SOTagsWantToWorkWith    72.345914
SOTagsAdmired           75.101819
SOTagsHaveEntry         99.085890
SOTagsWant Entry        99.137931
SOAccount               33.308444
SOVisitFreq             33.473618
SOPartFreq              34.464657
SOComm                  35.510001
SOFriction              37.354059
SO_Actions_1            46.596977
SO_Actions_16           50.549824
SO_Actions_3            46.596977
SO_Actions_4            46.596977
SO_Actions_6            46.596977
SO_Actions_7            46.596977
SO_Actions_15           46.596977
SO_Actions_15_TEXT      98.314327
dtype: float64

Unique values per feature:
SOTagsHaveWorkedWith    1691
SOTagsWantToWorkWith    2019
SOTagsAdmired           1070
SOTagsHaveEntry          313
SOTagsWant Entry         293
SOAccount                  3
SOVisitFreq                7
SOPartFreq                 8
SOComm                     6


In [ ]:
from scipy.stats import chi2_contingency

def check_significance(feature, target_col):
    # Drop NaNs for the test
    temp_df = df[[feature, target_col]].dropna()
    if temp_df.empty or temp_df[feature].nunique() < 2:
        return 1.0 # Not significant/Not testable

    contingency_table = pd.crosstab(temp_df[feature], temp_df[target_col])
    chi2, p, dof, ex = chi2_contingency(contingency_table)
    return p

# Perform Chi-Square tests for categorical features vs Job Satisfaction
significance_results = {}
for feature in my_features:
    if feature in df.columns:
        p_val = check_significance(feature, target)
        significance_results[feature] = p_val

sig_df = pd.DataFrame.from_dict(significance_results, orient='index', columns=['p-value'])
sig_df['Significant'] = sig_df['p-value'] < 0.05
display(sig_df.sort_values(by='p-value'))

,p-value,Significant
SOComm,1.222788e-20,True
SOPartFreq,3.303343e-12,True
SOFriction,1.028344e-05,True
SOVisitFreq,1.333633e-05,True
SOAccount,2.685114e-04,True
SO_Actions_3,3.642193e-02,True
SO_Actions_15,5.744336e-02,False
SO_Actions_7,8.299064e-02,False
SO_Actions_16,1.149481e-01,False
SO_Actions_4,1.465255e-01,False


### 2. Advanced Feature Selection: MI, Importance, and Multicollinearity
We will now run the model-based tests and information gain metrics to finalize our selection.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

# 1. Prepare data for advanced tests
# We drop the high-null columns first to make the model training viable
cols_to_test = [f for f in my_features if missing_info[f] < 90]

# Encode categorical variables for the model
encoded_df = df[cols_to_test + [target]].dropna().copy()
le = LabelEncoder()
for col in encoded_df.columns:
    encoded_df[col] = le.fit_transform(encoded_df[col].astype(str))

X = encoded_df[cols_to_test]
y = encoded_df[target]

# 2. Mutual Information
mi_scores = mutual_info_classif(X, y, discrete_features=True)
mi_results = pd.Series(mi_scores, index=cols_to_test).sort_values(ascending=False)

# 3. Random Forest Feature Importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)
rf_importance = pd.Series(rf.feature_importances_, index=cols_to_test).sort_values(ascending=False)

# 4. Multicollinearity (Correlation Matrix)
corr_matrix = X.corr().abs()

# Display Results
print("--- Mutual Information ---")
display(mi_results.head(10))
print("\n--- RF Feature Importance ---")
display(rf_importance.head(10))

--- Mutual Information ---


,0
SOTagsWantToWorkWith,0.232097
SOTagsHaveWorkedWith,0.179838
SOTagsAdmired,0.137678
SOComm,0.003380
SOPartFreq,0.002906
SO_Actions_15,0.002041
SO_Actions_3,0.001841
SO_Actions_6,0.001516
SO_Actions_4,0.001483
SO_Actions_16,0.001410



--- RF Feature Importance ---


,0
SOTagsWantToWorkWith,0.109199
SOTagsHaveWorkedWith,0.099828
SOTagsAdmired,0.090347
SO_Actions_6,0.079175
SO_Actions_3,0.078694
SO_Actions_7,0.078515
SO_Actions_1,0.077882
SO_Actions_4,0.071279
SO_Actions_16,0.067753
SOVisitFreq,0.061579


In [ ]:
# 5. Final Decision Table Generation
decision_data = []

for f in my_features:
    reason = ""
    status = "KEEP"

    # Logical/Quality Checks
    if missing_info[f] > 90:
        status = "REMOVE"
        reason = f"Too many missing values ({missing_info[f]:.1f}%)"
    elif f == 'SO_Actions_15_TEXT':
        status = "REMOVE"
        reason = "High cardinality free-text (Noise)"
    # Statistical Checks
    elif f in sig_df.index and not sig_df.loc[f, 'Significant']:
        # Check if MI or RF saved it
        if f in mi_results and mi_results[f] > 0.01:
            status = "KEEP"
            reason = "Passed MI threshold despite Chi-Square"
        else:
            status = "REMOVE"
            reason = f"Not statistically significant (p={sig_df.loc[f, 'p-value']:.3f})"
    else:
        reason = "Strong correlation and significance across multiple tests"

    decision_data.append({'Feature': f, 'Decision': status, 'Reason': reason})

final_summary = pd.DataFrame(decision_data)
display(final_summary)

,Feature,Decision,Reason
0,SOTagsHaveWorkedWith,KEEP,Passed MI threshold despite Chi-Square
1,SOTagsWantToWorkWith,KEEP,Passed MI threshold despite Chi-Square
2,SOTagsAdmired,KEEP,Passed MI threshold despite Chi-Square
3,SOTagsHaveEntry,REMOVE,Too many missing values (99.1%)
4,SOTagsWant Entry,REMOVE,Too many missing values (99.1%)
5,SOAccount,KEEP,Strong correlation and significance across mul...
6,SOVisitFreq,KEEP,Strong correlation and significance across mul...
7,SOPartFreq,KEEP,Strong correlation and significance across mul...
8,SOComm,KEEP,Strong correlation and significance across mul...
9,SOFriction,KEEP,Strong correlation and significance across mul...


In [ ]:
pd.set_option('display.max_colwidth', None)
display(final_summary[['Feature', 'Decision', 'Reason']])

,Feature,Decision,Reason
0,SOTagsHaveWorkedWith,KEEP,Passed MI threshold despite Chi-Square
1,SOTagsWantToWorkWith,KEEP,Passed MI threshold despite Chi-Square
2,SOTagsAdmired,KEEP,Passed MI threshold despite Chi-Square
3,SOTagsHaveEntry,REMOVE,Too many missing values (99.1%)
4,SOTagsWant Entry,REMOVE,Too many missing values (99.1%)
5,SOAccount,KEEP,Strong correlation and significance across multiple tests
6,SOVisitFreq,KEEP,Strong correlation and significance across multiple tests
7,SOPartFreq,KEEP,Strong correlation and significance across multiple tests
8,SOComm,KEEP,Strong correlation and significance across multiple tests
9,SOFriction,KEEP,Strong correlation and significance across multiple tests


### Why Mutual Information (MI) Overrode the p-value

**The 'Dilution' Problem (Chi-Square):**  
Columns like `SOTagsHaveWorkedWith` have very high cardinality (1,600+ unique combinations). Chi-Square tries to find a direct dependency across all these categories. Because the data is spread so thin, the test fails to find a simple relationship, leading to a high p-value (insignificant).

**The 'Information' Solution (MI):**  
Mutual Information calculates how much knowing the feature reduces uncertainty in the target. An MI score of 0.23 (for `SOTagsWantToWorkWith`) is actually very high for this kind of survey data. It indicates that the **technology stack** is one of the most informative indicators of satisfaction, even if the relationship isn't a simple one-to-one mapping.

### Updated Feature Selection Summary (Sorted by Importance)

**Summary Reason for Inclusion:** Features were kept if they showed high predictive signal through Mutual Information (for high-cardinality tech stacks) or demonstrated strong statistical significance ($p < 0.05$) and model importance in identifying Job Satisfaction patterns.

In [ ]:
# Consolidating and sorting the 'KEEP' features
# Ranking is based on a combination of MI scores and RF Importance

keep_features_data = [
    {"Feature": "SOTagsWantToWorkWith", "Rank": 1, "Reason": "Highest Mutual Information; tech stack interest is the strongest predictor.", "Metric": "MI: 0.23"},
    {"Feature": "SOTagsHaveWorkedWith", "Rank": 2, "Reason": "Significant information gain regarding current technical environment.", "Metric": "MI: 0.18"},
    {"Feature": "SOTagsAdmired", "Rank": 3, "Reason": "High non-linear correlation with professional aspirations and satisfaction.", "Metric": "MI: 0.13"},
    {"Feature": "SOComm", "Rank": 4, "Reason": "Highest statistical significance (p ~ 0); strong sense of community ties to satisfaction.", "Metric": "p < 0.001"},
    {"Feature": "SOPartFreq", "Rank": 5, "Reason": "Very high significance; active participation frequency is a key behavioral indicator.", "Metric": "p < 0.001"},
    {"Feature": "SOVisitFreq", "Rank": 6, "Reason": "Strong statistical relationship; regular platform utility correlates with work flow.", "Metric": "p < 0.001"},
    {"Feature": "SOAccount", "Rank": 7, "Reason": "Statistically significant; formal platform membership indicates professional engagement.", "Metric": "p < 0.001"},
    {"Feature": "SOFriction", "Rank": 8, "Reason": "Significant predictor; indicates how easily a developer finds solutions to work blockers.", "Metric": "p < 0.01"},
    {"Feature": "SO_Actions_3", "Rank": 9, "Reason": "Top performing sub-action; specific platform behaviors that align with satisfaction.", "Metric": "RF Imp: ~0.08"}
]

sorted_keep_df = pd.DataFrame(keep_features_data)
display(sorted_keep_df)

,Feature,Rank,Reason,Metric
0,SOTagsWantToWorkWith,1,Highest Mutual Information; tech stack interest is the strongest predictor.,MI: 0.23
1,SOTagsHaveWorkedWith,2,Significant information gain regarding current technical environment.,MI: 0.18
2,SOTagsAdmired,3,High non-linear correlation with professional aspirations and satisfaction.,MI: 0.13
3,SOComm,4,Highest statistical significance (p ~ 0); strong sense of community ties to satisfaction.,p < 0.001
4,SOPartFreq,5,Very high significance; active participation frequency is a key behavioral indicator.,p < 0.001
5,SOVisitFreq,6,Strong statistical relationship; regular platform utility correlates with work flow.,p < 0.001
6,SOAccount,7,Statistically significant; formal platform membership indicates professional engagement.,p < 0.001
7,SOFriction,8,Significant predictor; indicates how easily a developer finds solutions to work blockers.,p < 0.01
8,SO_Actions_3,9,Top performing sub-action; specific platform behaviors that align with satisfaction.,RF Imp: ~0.08


### Conclusion and Recommendations
Based on the analysis (Missing values, Cardinality, and Chi-Square p-values):

1. **Keep**: Features with p-value < 0.05 (e.g., usually `SOComm`, `SOVisitFreq`) as they show a statistical relationship with Job Satisfaction.
2. **Remove**: `SO_Actions_15_TEXT` and other free-text fields (too high cardinality/noise).
3. **Remove**: Features with > 50% missing values if they cannot be reasonably imputed.
4. **Keep/Feature Engineer**: `SOTags...` features can be converted to a 'count' of tags to see if engagement volume correlates with satisfaction.

**FINAL DECISIONS**
-----------------------------------

SOTagsHaveWorkedWith --> Keep --> Highest Mutual Information (tech stack you're working in) despite having a P-value > 0.05 as a one to one relation with the target column. Take care more than 50% of missing values

-----------------------------------

SOTagsWantToWorkWith --> Keep --> Significant information gain regarding current technical environment despite having a P-value > 0.05 as a one to one relation with the target column. Take care more than 70% of missing values

-------------------------------------

SOTagsAdmired --> Keep --> High non-linear correlation with professional aspirations and satisfaction. despite having a P-value > 0.05 as a one to one relation with the target column. Take care more than 70% of missing values

--------------------------------------

SOTagsHaveEntry --> Drop --> missing values more than 99% + p-value > 0.05

--------------------------------------

SOTagsWant Entry --> Drop --> missing
values more than 99% + p-value > 0.05

-------------------------------------

SOAccount --> Keep --> Strong correlation and significance across multiple tests. Strong statistical relationship. (low p-value < 0.05)

------------------------------------

SOVisitFreq --> Keep --> Strong correlation and significance across multiple tests. Strong statistical relationship. (low p-value < 0.05)

-------------------------------------

SOPartFreq --> Keep --> Strong correlation and significance across multiple tests. Very high statistical significance (2nd lowest p-value)

-------------------------------------

SO_Actions_1 --> Drop --> P-value > 0.05 which doesn't deny the hypothesis of the feature having no correlation to the target

-------------------------------------

SO_Actions_16 --> Drop --> P-value > 0.05 which doesn't deny the hypothesis of the feature having no correlation to the target + having 50% missing values

-------------------------------------

SO_Actions_3 --> Keep --> Strong correlation and significance across multiple tests. Strong statistical relationship. (low p-value < 0.05)

--------------------------------------

SO_Actions_4 --> Drop --> P-value > 0.05 which doesn't deny the hypothesis of the feature having no correlation to the target

-------------------------------------

SO_Actions_6 --> Drop --> P-value > 0.05 which doesn't deny the hypothesis of the feature having no correlation to the target

-------------------------------------

SO_Actions_7 --> Drop --> P-value > 0.05 which doesn't deny the hypothesis of the feature having no correlation to the target

------------------------------------

SO_Actions_15 --> Drop --> P-value > 0.05 which doesn't deny the hypothesis of the feature having no correlation to the target

------------------------------------

SO_Actions_15_TEXT --> drop --> missing values more than 98% + high variance which can lead to overfitting + p-value > 0.05

-----------------------------------

SOComm --> Keep --> Strong correlation and significance across multiple tests. Highest statistical significance (lowest p-value)

-----------------------------------

SOFriction --> Keep --> Strong correlation and significance across multiple tests. Strong statistical relationship. (low p-value < 0.05)

---------------------------------